# Semantic Federated — Experimentos no Colab

Notebook pensado para ser aberto direto pelo GitHub (**Arquivo > Abrir notebook > GitHub**, colando a URL do repositório `GianSE/semantic-federeted`).

Antes de rodar: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4)**.

Fluxo das células:
1. Checar GPU
2. Clonar o repositório (o Colab abre só o `.ipynb`, não o resto do repo — por isso o clone é necessário)
3. Instalar dependências
4. Montar o Google Drive e redirecionar `data/` e `results/` para lá (persistência entre sessões, já que o disco do Colab é efêmero)
5. Rodar os 3 blocos de experimento (grid principal multi-seed, não-IID, canais com desvanecimento)
6. Conferir os resultados e as 6 figuras geradas em `results/plots/`

In [ ]:
!nvidia-smi

Se a célula acima der erro (`command not found` / `no devices found`), o ambiente de execução não está configurado como GPU — ajuste em **Ambiente de execução > Alterar tipo de ambiente de execução** antes de continuar.

In [ ]:
REPO_URL = "https://github.com/GianSE/semantic-federeted.git"
REPO_DIR = "/content/semantic-federeted"

import os

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repositorio ja clonado, pulando.")

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## Persistência via Google Drive

O disco do Colab é apagado quando a sessão termina. Para não perder os datasets baixados (evita re-download a cada sessão) nem os resultados acumulados (`results/`), apontamos essas duas pastas para o seu Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/semantic-federeted"
os.makedirs(f"{DRIVE_BASE}/data", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/results", exist_ok=True)

import shutil

for local_name, drive_path in [("data", f"{DRIVE_BASE}/data"), ("results", f"{DRIVE_BASE}/results")]:
    local_path = os.path.join(REPO_DIR, local_name)
    if os.path.islink(local_path):
        continue
    if os.path.isdir(local_path):
        shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f"{local_name}/ -> {drive_path}")

## Rodar os experimentos

O `main.py` já salva os resultados incrementalmente (um `save_results` por configuração) e regenera `results/tables/results_summary.csv` + as figuras em `results/plots/` ao final de cada execução. Se a sessão cair no meio, os resultados já salvos continuam no Drive e dá para rodar de novo depois — nada é sobrescrito.

São 3 blocos, cada um alimentando um conjunto diferente de figuras:
1. **Grid principal** (multi-seed, AWGN, IID) — gera `accuracy_vs_compression_ratio.png`, `accuracy_vs_latent_dim.png` (com ajuste log), `accuracy_vs_noise_level.png` e `communication_cost_vs_latent_dim.png`.
2. **Não-IID (Dirichlet)** — gera `accuracy_iid_vs_dirichlet.png` (precisa dos dados do bloco 1 para ter o lado "iid" da comparação).
3. **Canais com desvanecimento (Rayleigh/Rician)** — gera `accuracy_by_channel_type.png` (idem, precisa do lado "awgn" do bloco 1).

Rode os 3 em sequência (ou em sessões separadas, já que tudo acumula no mesmo CSV).

In [ ]:
# Bloco 1 — grid principal: compressao, ruido e dimensao latente, multi-seed, AWGN/IID.
!python main.py \
  --datasets cifar10 \
  --latent-dims 16 32 64 128 \
  --noise-levels 0.0 0.01 0.05 0.1 \
  --num-clients 5 \
  --rounds 3 \
  --seeds 1 2 3 4 5 \
  --num-workers 2

### Bloco 2 — não-IID (Dirichlet)

Mesma dimensão latente (L=64) do bloco 1, para comparar diretamente IID vs. não-IID em `accuracy_iid_vs_dirichlet.png`. `--dirichlet-alpha` baixo = mais heterogêneo entre clientes.

In [ ]:
# Bloco 2 — particao nao-IID (Dirichlet), canal AWGN (igual ao bloco 1) para isolar
# o efeito da heterogeneidade dos dados.
!python main.py \
  --datasets cifar10 \
  --latent-dims 64 \
  --noise-levels 0.0 0.05 \
  --num-clients 5 \
  --rounds 3 \
  --seeds 1 2 3 4 5 \
  --partition dirichlet \
  --dirichlet-alpha 0.3 \
  --num-workers 2

### Bloco 3 — canais com desvanecimento (Rayleigh / Rician)

Mesma dimensão latente (L=64) e partição IID do bloco 1, para comparar AWGN vs. Rayleigh vs. Rician em `accuracy_by_channel_type.png`. São duas execuções (um canal por vez).

In [ ]:
# Bloco 3 — canal Rayleigh (sem linha de visada).
!python main.py \
  --datasets cifar10 \
  --latent-dims 64 \
  --noise-levels 0.0 0.05 \
  --num-clients 5 \
  --rounds 3 \
  --seeds 1 2 3 4 5 \
  --channel-type rayleigh \
  --fading-scale 1.0 \
  --num-workers 2

# Bloco 3 — canal Rician (com linha de visada, K=5).
!python main.py \
  --datasets cifar10 \
  --latent-dims 64 \
  --noise-levels 0.0 0.05 \
  --num-clients 5 \
  --rounds 3 \
  --seeds 1 2 3 4 5 \
  --channel-type rician \
  --fading-scale 1.0 \
  --rician-k 5.0 \
  --num-workers 2

## Conferir resultados

In [ ]:
import pandas as pd

df = pd.read_csv("results/data/experiment_results.csv")
df

In [ ]:
import os
from IPython.display import Image, display

plots_dir = "results/plots"
expected = [
    "accuracy_vs_compression_ratio.png",
    "accuracy_vs_latent_dim.png",
    "accuracy_vs_noise_level.png",
    "communication_cost_vs_latent_dim.png",
    "accuracy_iid_vs_dirichlet.png",
    "accuracy_by_channel_type.png",
]
for name in expected:
    path = os.path.join(plots_dir, name)
    if os.path.exists(path):
        print(name)
        display(Image(filename=path))
    else:
        print(f"{name} -- ainda nao gerado (rode o bloco correspondente)")

Os gráficos (`results/plots/`) e tabelas LaTeX (`results/tables/`) ficam disponíveis no Google Drive em `MyDrive/semantic-federeted/results/`, prontos para serem baixados ou referenciados direto no paper.